In [ ]:
import pandas as pd

df = pd.read_csv('../metadata/SraRunTable.csv', header = 0)

In [ ]:
def generate_sample_names(samples_df, naming_scheme='sequential'):
    """
    Generate sample names based on the chosen naming scheme
    
    Args:
        samples_df: DataFrame with grouped samples
        naming_scheme: 'gsm', 'tumor_type', or 'sequential'
    
    Returns:
        Dictionary mapping GSM IDs to sample names
    """
    sample_mapping = {}
    
    if naming_scheme == 'gsm':
        for _, row in samples_df.iterrows():
            sample_mapping[row['Sample Name']] = row['Sample Name']
    
    elif naming_scheme == 'tumor_type':
        for _, row in samples_df.iterrows():
            gsm = row['Sample Name']
            tumor_type = row['tumor_type'].replace(' ', '_')
            age = str(row['age']).replace(' ', '_')
            sample_name = f"{tumor_type}_{age}"
            
            # Handle duplicates by adding suffix
            base_name = sample_name
            counter = 1
            while sample_name in sample_mapping.values():
                sample_name = f"{base_name}_{counter}"
                counter += 1
            
            sample_mapping[gsm] = sample_name
    
    elif naming_scheme == 'sequential':
        # Sort by tumor_type
        samples_sorted = samples_df.sort_values(['tumor_type'], ascending=[True])
        
        # Count samples per tumor_type
        tumor_type_counts = {}
        
        for _, row in samples_sorted.iterrows():
            gsm = row['Sample Name']
            tumor_type = row['tumor_type'].replace(' ', '_')
            
            if tumor_type not in tumor_type_counts:
                tumor_type_counts[tumor_type] = 0
            tumor_type_counts[tumor_type] += 1
            
            sample_name = f"{tumor_type}_{tumor_type_counts[tumor_type]:03d}"
            sample_mapping[gsm] = sample_name
    
    return sample_mapping

In [ ]:
def generate_samplesheet(metadata_file, fastq_base_dir, output_file, naming_scheme='sequential'):
    """
    Generate samplesheet grouping runs by GSM ID
    
    Args:
        metadata_file: Path to SRA metadata CSV file
        fastq_base_dir: Base directory where FASTQ files are stored
        output_file: Output samplesheet CSV file
        naming_scheme: How to name samples ('gsm', 'tumor_type', or 'sequential')
    """
    
    # Read metadata
    df = pd.read_csv(metadata_file)
    
    def safe_agg(x, field_name):
        """Aggregate by taking first value, but validate all are the same"""
        unique_vals = x.unique()
        if len(unique_vals) > 1:
            print(f"WARNING: Inconsistent {field_name} values found: {unique_vals}")
            print(f"         Using first value: {unique_vals[0]}")
        return unique_vals[0]
    
    # Group by Sample Name (GSM ID) to get unique biological samples
    samples = df.groupby('Sample Name').agg({
        'Run': lambda x: list(x),
        'age': lambda x: safe_agg(x, 'age'),
        'tissue': lambda x: safe_agg(x, 'tissue'),
        'tumor_type': lambda x: safe_agg(x, 'tumor_type'),
        'source_name': lambda x: safe_agg(x, 'source_name'),
        'cell_type': lambda x: safe_agg(x, 'cell_type')
    }).reset_index()
    
    # Generate sample names based on scheme
    sample_mapping = generate_sample_names(samples, naming_scheme)
    
    # Create samplesheet
    samplesheet_data = []
    
    for _, row in samples.iterrows():
        gsm_id = row['Sample Name']
        sample_id = sample_mapping[gsm_id]
        runs = row['Run']
        
        fastq_dir = f"{fastq_base_dir}/{sample_id}"
        
        samplesheet_data.append({
            'sample_id': sample_id,
            'gsm_id': gsm_id,
            'fastq_dir': fastq_dir,
            'n_runs': len(runs),
            'runs': ','.join(runs),
            'age': row['age'],
            'tissue': row['tissue'],
            'tumor_type': row['tumor_type'],
            'source_name': row['source_name'],
            'cell_type': row['cell_type']
        })
    
    # Create DataFrame and save
    samplesheet_df = pd.DataFrame(samplesheet_data)
    
    # Save full version with metadata
    samplesheet_df.to_csv(output_file.replace('.csv', '_full.csv'), index=False)
    print(f"Full samplesheet saved to: {output_file.replace('.csv', '_full.csv')}")
    
    # Save minimal version for pipeline
    minimal_df = samplesheet_df[['sample_id', 'fastq_dir']]
    minimal_df.to_csv(output_file, index=False)
    print(f"Minimal samplesheet saved to: {output_file}")
    
    # Print summary
    print(f"\nSummary:")
    print(f"Total biological samples: {len(samplesheet_df)}")
    print(f"Total sequencing runs: {samplesheet_df['n_runs'].sum()}")
    print(f"\nRuns per sample:")
    print(samplesheet_df['n_runs'].value_counts().sort_index())
    print(f"\nSample breakdown by tumor type:")
    print(samplesheet_df['tumor_type'].value_counts())
    
    return samplesheet_df

In [ ]:
df = generate_samplesheet('../metadata/SraRunTable.csv','fastq/cellranger','../metadata/cellranger_samples.csv','sequential')

In [ ]:
def create_download_script(metadata_file, output_script, naming_scheme='tumor_type'):
    """
    Create a bash script to download FASTQ files organized by sample
    
    Args:
        naming_scheme: How to name samples
            - 'gsm': Use GSM IDs (e.g., GSM6619237)
            - 'tumor_type': Use tumor_type_age (e.g., primary_tumor_67_years)
            - 'sequential': Use tumor_type with numbers (e.g., primary_tumor_001, recurrent_tumor_001)
    """
    
    df = pd.read_csv(metadata_file)
    
    # Group by Sample Name
    sample_groups = df.groupby('Sample Name').agg({
        'Run': lambda x: list(x),
        'age': lambda x: x.iloc[0],
        'tissue': lambda x: x.iloc[0],
        'tumor_type': lambda x: x.iloc[0],
        'source_name': lambda x: x.iloc[0]
    }).reset_index()
    
    # Generate sample names based on scheme
    sample_mapping = generate_sample_names(sample_groups, naming_scheme)
    
    with open(output_script, 'w') as f:
        f.write("#!/bin/bash\n")
        f.write("""
#SBATCH -J Spatial
#SBATCH -p gpu
#SBATCH -o spatial_%j.txt
#SBATCH -e spatial_%j.err
#SBATCH --mail-type=ALL
#SBATCH --mail-user=sidrajes@iu.edu
#SBATCH --nodes=1
#SBATCH --ntasks-per-node=4
#SBATCH --cpus-per-task=4
#SBATCH --time=18:00:00
#SBATCH --mem=30GB
#SBATCH -A r00750

base=/N/project/Krolab/Siddharth/Pipelines/spatial
cd $base

module load sra-toolkit
""")
        f.write("# Download and organize FASTQ files by sample\n")
        f.write("# Files are renamed to Cell Ranger format\n")
        f.write(f"# Naming scheme: {naming_scheme}\n")
        f.write("# Uses SRA Toolkit (prefetch + fasterq-dump)\n\n")
        f.write("set -euo pipefail\n\n")
        f.write("# Check if required tools are installed\n")
        f.write("command -v prefetch >/dev/null 2>&1 || { echo 'prefetch not found. Install SRA Toolkit.'; exit 1; }\n")
        f.write("command -v fasterq-dump >/dev/null 2>&1 || { echo 'fasterq-dump not found. Install SRA Toolkit.'; exit 1; }\n")
        f.write("command -v gzip >/dev/null 2>&1 || { echo 'gzip not found. Install gzip for compression.'; exit 1; }\n\n")
        
        # Write mapping table as comment
        f.write("# Sample Mapping:\n")
        for gsm, sample_name in sample_mapping.items():
            f.write(f"# {gsm} -> {sample_name}\n")
        f.write("\n")
        
        for _, row in sample_groups.iterrows():
            gsm = row['Sample Name']
            sample_name = sample_mapping[gsm]
            runs = row['Run']
            
            f.write(f"\n# {'='*60}\n")
            f.write(f"# Sample: {sample_name} (Original: {gsm})\n")
            f.write(f"# Tumor type: {row['tumor_type']}, Age: {row['age']}\n")
            f.write(f"# {'='*60}\n\n")
            
            f.write(f"echo 'Processing {sample_name}...'\n")
            f.write(f"mkdir -p fastq/cellranger/{sample_name}\n")
            f.write(f"cd fastq/cellranger/{sample_name}\n\n")
            
            for i, run in enumerate(runs, 1):
                f.write(f"# Run {i}: {run}\n")
                f.write(f"echo '  Downloading {run}...'\n")
                f.write(f"prefetch {run} || {{ echo 'Failed to prefetch {run}'; exit 1; }}\n")
                f.write(f"fasterq-dump {run} --split-files --include-technical --threads 8 || {{ echo 'Failed to dump {run}'; exit 1; }}\n\n")
                
                f.write(f"# Detect read layout and rename to Cell Ranger format:\n")
                f.write(f"# 4 files (_1,_2,_3,_4): dual-index 10x  -> _3=R1 (barcode+UMI), _4=R2 (cDNA)\n")
                f.write(f"# 3 files (_1,_2,_3):    single-index 10x -> _2=R1 (barcode+UMI), _3=R2 (cDNA)\n")
                f.write(f"# 2 files (_1,_2):        standard paired  -> _1=R1, _2=R2\n")
                f.write(f"if [ -f {run}_4.fastq ]; then\n")
                f.write(f"    echo '  4-file layout: using _3=R1, _4=R2'\n")
                f.write(f"    mv {run}_3.fastq {sample_name}_S1_L00{i}_R1_001.fastq\n")
                f.write(f"    mv {run}_4.fastq {sample_name}_S1_L00{i}_R2_001.fastq\n")
                f.write(f"    rm -f {run}_1.fastq {run}_2.fastq\n")
                f.write(f"elif [ -f {run}_3.fastq ]; then\n")
                f.write(f"    echo '  3-file layout: using _2=R1, _3=R2'\n")
                f.write(f"    mv {run}_2.fastq {sample_name}_S1_L00{i}_R1_001.fastq\n")
                f.write(f"    mv {run}_3.fastq {sample_name}_S1_L00{i}_R2_001.fastq\n")
                f.write(f"    rm -f {run}_1.fastq\n")
                f.write(f"elif [ -f {run}_1.fastq ] && [ -f {run}_2.fastq ]; then\n")
                f.write(f"    echo '  2-file layout: using _1=R1, _2=R2'\n")
                f.write(f"    mv {run}_1.fastq {sample_name}_S1_L00{i}_R1_001.fastq\n")
                f.write(f"    mv {run}_2.fastq {sample_name}_S1_L00{i}_R2_001.fastq\n")
                f.write(f"else\n")
                f.write(f"    echo 'ERROR: Cannot find expected FASTQ files for {run}'\n")
                f.write(f"    ls -lh {run}*.fastq 2>/dev/null || echo 'No fastq files found'\n")
                f.write(f"    exit 1\n")
                f.write(f"fi\n\n")
                
                f.write(f"echo '  Compressing...'\n")
                f.write(f"pigz {sample_name}_S1_L00{i}_R1_001.fastq\n")
                f.write(f"pigz {sample_name}_S1_L00{i}_R2_001.fastq\n\n")
                f.write(f"rm -rf {run}\n\n")
            
            f.write(f"cd ../../..\n")
            f.write(f"echo 'Completed {sample_name}'\n")
            f.write(f"echo ''\n\n")
        
        f.write("\necho 'All samples downloaded and renamed!'\n")
        f.write("echo 'Directory structure:'\n")
        f.write("tree fastq/ -L 2\n")
    
    # Save sample mapping to CSV
    mapping_df = pd.DataFrame([
        {'gsm_id': gsm, 'sample_name': name, **sample_groups[sample_groups['Sample Name'] == gsm].iloc[0].to_dict()}
        for gsm, name in sample_mapping.items()
    ])
    mapping_df = mapping_df[['gsm_id', 'sample_name', 'tumor_type', 'age', 'tissue', 'source_name']]
    mapping_file = output_script.replace('.sh', '_mapping.csv')
    mapping_df.to_csv(mapping_file, index=False)
    
    print(f"Download script saved to: {output_script}")
    print(f"Sample mapping saved to: {mapping_file}")
    print(f"Make executable with: chmod +x {output_script}")

In [ ]:
create_download_script('../metadata/SraRunTable.csv', '../metadata/download_cellranger.sh', 'sequential')